# Lab #28 — Full Platform Integration Sprint

## AI Platform Hybrid Architecture (Local + Kaggle GPU)

This notebook demonstrates the complete AI platform integration:
- **Local Stack (Docker)**: Kafka, Prefect, Qdrant, Redis, Prometheus, Grafana, API Gateway
- **Kaggle GPU**: vLLM serving (T4) / Transformers fallback (P100), Embedding service, MLflow

### GPU Compatibility

| GPU | Compute Capability | Recommended Serving |
|-----|-------------------|---------------------|
| T4 | sm_75 | **vLLM** (fast, optimized) |
| P100 | sm_60 | **Transformers** (via `device_map="auto"`) |

The notebook **auto-detects GPU** and uses vLLM if available (T4), otherwise falls back to transformers (P100).

### Prerequisites
- Docker Desktop running with all services up
- Kaggle account with GPU (T4 or P100) enabled
- ngrok token from ngrok.com

## Step 1: Install Dependencies

In [ ]:
# Cell 1: Install dependencies
# - vLLM: requires GPU compute capability sm_75+ (T4 ✓, P100 ✗)
# - transformers: works on all CUDA GPUs (T4 ✓, P100 ✓)
# The server code auto-detects GPU and picks the right approach.

!pip install -q vllm transformers==4.46.3 accelerate fastapi uvicorn pyngrok mlflow sentence-transformers requests scipy

# Fallback packages if vLLM fails on P100 (transformers is already installed above)
print("Dependencies installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 81.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.1/264.1 MB 7.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.2 MB/s eta 0:00:00:00:0100:02
   ━━━━━━━━━━━━

## Step 2: Configure ngrok Tunnel

Set your ngrok auth token below (get from [ngrok.com](https://ngrok.com)).

In [5]:
# Cell 2: Setup ngrok
# Replace "YOUR_NGROK_TOKEN" with your actual token from https://ngrok.com
from pyngrok import ngrok

NGROK_TOKEN = "3DtO591hKttikzGAN97n04nKKe9_7F4phyBYy626wUwQcuhVT"  # <-- Replace this!
ngrok.set_auth_token(NGROK_TOKEN)
print("ngrok configured successfully")

ngrok configured successfully


## Step 3: Start Model Server (vLLM for T4 / Transformers for P100)

This cell **auto-detects your GPU** and chooses the best serving method:
- **T4 (sm_75+)**: Uses **vLLM** — fast, optimized inference
- **P100 (sm_60)**: Uses **Transformers** — compatible fallback

The server listens on **port 8001** and exposes the OpenAI-compatible `/v1/chat/completions` endpoint.

In [3]:
# Cell 3: Auto-detect GPU and start model server
import subprocess, threading, time, sys

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4"
PORT = 8001

# ── GPU Detection ────────────────────────────────────────────────────────────
import torch
gpu_name = torch.cuda.get_device_name(0)
gpu_cc = torch.cuda.get_device_capability(0)
print(f"Detected GPU: {gpu_name} (compute capability sm_{gpu_cc[0]}{gpu_cc[1]})")

USE_VLLM = gpu_cc[0] >= 7  # vLLM requires sm_70+
if USE_VLLM:
    print("Decision: vLLM (fast, optimized inference)")
else:
    print("Decision: Transformers fallback (P100 — vLLM not compatible with sm_60)")

# ── Build server script ───────────────────────────────────────────────────────
if USE_VLLM:
    # vLLM server (T4)
    server_script = f"""
import subprocess, sys
sys.stdout.flush()
sys.stderr.flush()
subprocess.run([
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", "{MODEL_NAME}",
    "--port", "{PORT}",
    "--max-model-len", "4096",
    "--gpu-memory-utilization", "0.5"
])
"""
else:
    # Transformers + FastAPI server (P100)
    server_script = f"""
import os, sys, torch
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import uvicorn

app = FastAPI(title="Qwen2.5-7B API (Transformers)", version="1.0.0")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading model on " + DEVICE + "...")
tokenizer = AutoTokenizer.from_pretrained("{MODEL_NAME}", trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    "{MODEL_NAME}",
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
print("Model loaded!")

class ChatRequest(BaseModel):
    model: str
    messages: list[dict]
    max_tokens: int = 512
    temperature: float = 0.7

@app.get("/health")
def health():
    return JSONResponse({{"status": "ok", "engine": "transformers"}})

@app.post("/v1/chat/completions")
def chat_completions(req: ChatRequest):
    prompt = ""
    for msg in req.messages:
        role = msg.get("role", "user")
        content = msg.get("content", "")
        if role == "system": prompt += f"System: {{content}}\\n"
        elif role == "user":   prompt += f"User: {{content}}\\n"
        else:                  prompt += f"Assistant: {{content}}\\n"
    prompt += "Assistant:"
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    outputs = model.generate(
        **inputs,
        max_new_tokens=req.max_tokens,
        temperature=req.temperature,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return {{
        "model": req.model,
        "choices": [{{"message": {{"role": "assistant", "content": response.strip()}}}}],
    }}

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port={PORT}, log_level="warning")
"""

# Write server script to temp file
with open("/tmp/model_server.py", "w") as f:
    f.write(server_script)
print(f"Server script written ({'vLLM' if USE_VLLM else 'Transformers'} mode)")

# ── Start server in background thread ─────────────────────────────────────────
def run_server():
    subprocess.run([sys.executable, "/tmp/model_server.py"])

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
print(f"Model server starting on port {PORT}...")

# Wait for model to load
wait_time = 90 if USE_VLLM else 120
print(f"Waiting {wait_time}s for model to load...")
time.sleep(wait_time)

# Verify server is up
import requests as _req
for attempt in range(6):
    try:
        r = _req.get(f"http://localhost:{PORT}/health", timeout=15)
        if r.status_code == 200:
            print(f"Server is healthy! {r.json()}")
            break
    except:
        pass
    time.sleep(15)
else:
    print("WARNING: Server may still be loading — check /health endpoint manually")

Detected GPU: Tesla T4 (compute capability sm_75)
Decision: vLLM (fast, optimized inference)
Server script written (vLLM mode)
Model server starting on port 8001...
Waiting 90s for model to load...


2026-05-18 10:09:40.993672: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779098981.213707     176 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779098981.273276     176 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779098981.817972     176 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779098981.818013     176 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779098981.818016     176 computation_placer.cc:177] computation placer alr

INFO 05-18 10:09:58 __init__.py:183] Automatically detected platform cuda.
INFO 05-18 10:10:00 api_server.py:835] vLLM API server version 0.7.0
INFO 05-18 10:10:00 api_server.py:836] args: Namespace(host=None, port=8001, uvicorn_log_level='info', allow_credentials=False, allowed_origins=['*'], allowed_methods=['*'], allowed_headers=['*'], api_key=None, lora_modules=None, prompt_adapters=None, chat_template=None, chat_template_content_format='auto', response_role='assistant', ssl_keyfile=None, ssl_certfile=None, ssl_ca_certs=None, ssl_cert_reqs=0, root_path=None, middleware=[], return_tokens_as_token_ids=False, disable_frontend_multiprocessing=False, enable_request_id_headers=False, enable_auto_tool_choice=False, tool_call_parser=None, tool_parser_plugin='', model='Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4', task='auto', tokenizer=None, skip_tokenizer_init=False, revision=None, code_revision=None, tokenizer_revision=None, tokenizer_mode='auto', trust_remote_code=False, allowed_local_media_path

2026-05-18 10:10:07.056500: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779099007.077641     196 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779099007.084393     196 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779099007.101768     196 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779099007.101797     196 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779099007.101800     196 computation_placer.cc:177] computation placer alr

INFO 05-18 10:10:13 __init__.py:183] Automatically detected platform cuda.
INFO 05-18 10:10:18 config.py:520] This model supports multiple tasks: {'classify', 'embed', 'score', 'reward', 'generate'}. Defaulting to 'generate'.
WARNING 05-18 10:10:19 config.py:599] gptq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
INFO 05-18 10:10:31 config.py:520] This model supports multiple tasks: {'reward', 'embed', 'score', 'generate', 'classify'}. Defaulting to 'generate'.
WARNING 05-18 10:10:32 config.py:599] gptq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
INFO 05-18 10:10:32 llm_engine.py:232] Initializing an LLM engine (v0.7.0) with config: model='Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=

...

INFO 05-18 10:11:02 model_runner.py:1115] Loading model weights took 5.2036 GB
INFO 05-18 10:11:05 worker.py:266] Memory profiling takes 2.96 seconds
INFO 05-18 10:11:05 worker.py:266] the current vLLM instance can use total_gpu_memory (14.56GiB) x gpu_memory_utilization (0.50) = 7.28GiB
INFO 05-18 10:11:05 worker.py:266] model weights take 5.20GiB; non_torch_memory takes 0.05GiB; PyTorch activation peak memory takes 1.42GiB; the rest of the memory reserved for KV Cache is 0.61GiB.
INFO 05-18 10:11:05 executor_base.py:108] # CUDA blocks: 718, # CPU blocks: 4681
INFO 05-18 10:11:05 executor_base.py:113] Maximum concurrency for 4096 tokens per request: 2.80x
INFO 05-18 10:11:10 model_runner.py:1430] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_utiliza

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:41<00:00,  1.18s/it]


INFO 05-18 10:11:52 model_runner.py:1558] Graph capturing finished in 41 secs, took 0.47 GiB
INFO 05-18 10:11:52 llm_engine.py:429] init engine (profile, create kv cache, warmup model) took 50.29 seconds
INFO 05-18 10:11:52 api_server.py:753] Using supplied chat template:
INFO 05-18 10:11:52 api_server.py:753] None
INFO 05-18 10:11:52 launcher.py:19] Available routes are:
INFO 05-18 10:11:52 launcher.py:27] Route: /openapi.json, Methods: HEAD, GET
INFO 05-18 10:11:52 launcher.py:27] Route: /docs, Methods: HEAD, GET
INFO 05-18 10:11:52 launcher.py:27] Route: /docs/oauth2-redirect, Methods: HEAD, GET
INFO 05-18 10:11:52 launcher.py:27] Route: /redoc, Methods: HEAD, GET
INFO 05-18 10:11:52 launcher.py:27] Route: /health, Methods: GET
INFO 05-18 10:11:52 launcher.py:27] Route: /ping, Methods: GET, POST
INFO 05-18 10:11:52 launcher.py:27] Route: /tokenize, Methods: POST
INFO 05-18 10:11:52 launcher.py:27] Route: /detokenize, Methods: POST
INFO 05-18 10:11:52 launcher.py:27] Route: /v1/model

INFO:     Started server process [176]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


INFO:     ::1:37226 - "GET /health HTTP/1.1" 200 OK
INFO:     ::1:44042 - "GET /health HTTP/1.1" 200 OK


## Step 4: Create ngrok Tunnel

Expose the model server to the internet via ngrok. **Copy the `public_url` from the output below**.

In [6]:
# Cell 4: Create ngrok tunnel
tunnel = ngrok.connect(8001, "http")
VLLM_PUBLIC_URL = tunnel.public_url
print(f"Model server public URL: {VLLM_PUBLIC_URL}")
print()
print("=" * 65)
print("ACTION REQUIRED — Update .env on your LOCAL machine:")
print()
print("1. On your LOCAL machine (not Kaggle), open: .env")
print("2. Find the line:")
print("     VLLM_NGROK_URL=https://your-vllm-url.ngrok-free.app")
print()
print(f"3. Replace with:  VLLM_NGROK_URL={VLLM_PUBLIC_URL}")
print()
print("4. Save the .env file — then continue with smoke tests on local")
print("=" * 65)

Model server public URL: https://kinswoman-wiry-clasp.ngrok-free.dev

ACTION REQUIRED — Update .env on your LOCAL machine:

1. On your LOCAL machine (not Kaggle), open: .env
2. Find the line:
     VLLM_NGROK_URL=https://your-vllm-url.ngrok-free.app

3. Replace with:  VLLM_NGROK_URL=https://kinswoman-wiry-clasp.ngrok-free.dev

4. Save the .env file — then continue with smoke tests on local


## Step 5: Copy ngrok URL to Local `.env`

**On your LOCAL machine (NOT on Kaggle):**

1. Look at the output from the cell above — copy the URL (e.g. `https://abc123.ngrok-free.app`)
2. Open the `.env` file in this repo on your local machine
3. Find: `VLLM_NGROK_URL=https://your-vllm-url.ngrok-free.app`
4. Replace with your actual URL
5. Save the file

Then continue to the next cells for smoke tests and production readiness.

## Step 6: Observability Verification

In [7]:
# Cell: Observability verification
import requests

print("=== Observability Verification ===\n")

# Prometheus metrics
try:
    resp = requests.get(
        "http://localhost:9090/api/v1/query",
        params={"query": 'up{job="api-gateway"}'},
        timeout=5
    )
    data = resp.json()
    assert data["status"] == "success"
    print("[PASS] Prometheus: Metrics flowing from API Gateway")
except Exception as e:
    print(f"[FAIL] Prometheus: {e}")

# Prometheus metrics endpoint
try:
    resp = requests.get("http://localhost:8000/metrics", timeout=5)
    assert resp.status_code == 200
    print("[PASS] Prometheus: /metrics endpoint exposed")
except Exception as e:
    print(f"[FAIL] Metrics endpoint: {e}")

# Grafana
try:
    resp = requests.get(
        "http://localhost:3000/api/health",
        auth=("admin", "admin"),
        timeout=5
    )
    assert resp.status_code == 200
    version = resp.json().get("version", "unknown")
    print(f"[PASS] Grafana: Healthy (version={version})")
except Exception as e:
    print(f"[FAIL] Grafana: {e}")

print("\n=== Done ===")

=== Observability Verification ===

[FAIL] Prometheus: HTTPConnectionPool(host='localhost', port=9090): Max retries exceeded with url: /api/v1/query?query=up%7Bjob%3D%22api-gateway%22%7D (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7bf49b70c890>: Failed to establish a new connection: [Errno 111] Connection refused'))
[FAIL] Metrics endpoint: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /metrics (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7bf49b70d070>: Failed to establish a new connection: [Errno 111] Connection refused'))
[FAIL] Grafana: HTTPConnectionPool(host='localhost', port=3000): Max retries exceeded with url: /api/health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7bf49b70d940>: Failed to establish a new connection: [Errno 111] Connection refused'))

=== Done ===


## Kafka Ingestion (Local)

Ingest sample data into Kafka topic `data.raw`. Run on your **local machine** (not in Kaggle):

In [ ]:
# Kafka Ingestion — Run on LOCAL machine:
#   cd D:\Project_AI\Day28-Lab-Assignment
#   python scripts/01_ingest_to_kafka.py
#
# This sends 5 sample documents to the Kafka topic 'data.raw'.
# The Prefect flow will then consume them and save to Delta Lake.

## Step 7: Production Readiness Check

Run the full production readiness checklist. Target: **Score >80%**.

In [ ]:
# Production Readiness Check — Run on LOCAL machine:
#
#   cd D:\Project_AI\Day28-Lab-Assignment
#   python scripts/production_readiness_check.py
#
# Expected output:
#   Overall Score:   X/16 = XX%
#   Critical Score:  X/XX = XX%
#   Status:          READY (target >80%)

## Deploy Prefect Flows (Local)

Deploy the Prefect flow that consumes from Kafka and saves to Delta Lake. Run on your **local machine**:

```bash
cd prefect/flows
pip install -r requirements.txt
python kafka_to_delta.py
```

This registers and deploys the `kafka-to-delta` flow to the Prefect work pool `lab28-pool`.

## Step 8: Smoke Tests

Run end-to-end smoke tests against the local Docker stack. Execute this in your **local terminal** (not in Kaggle), after Docker services are up.

**On your LOCAL machine, run:**
```
pytest smoke-tests/ -v
```

**Expected: 5/5 tests passing**

## Step 9: Test Inference Endpoint

Test the model server (after it has loaded and ngrok tunnel is active).

In [ ]:
# Cell: Test inference via local model server (port 8001)
import requests, time

PORT = 8001
BASE_URL = f"http://localhost:{PORT}"

print(f"Testing model server on port {PORT}...\n")

# Test 1: Health check
try:
    resp = requests.get(f"{BASE_URL}/health", timeout=10)
    print(f"[1] Health check: {resp.status_code} — {resp.json()}")
except Exception as e:
    print(f"[1] Health check FAILED: {e}")

# Test 2: Chat completions
try:
    resp = requests.post(
        f"{BASE_URL}/v1/chat/completions",
        json={
            "model": "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4",
            "messages": [{"role": "user", "content": "What is platform engineering in 2 sentences?"}],
            "max_tokens": 128,
            "temperature": 0.7
        },
        timeout=180
    )
    if resp.status_code == 200:
        result = resp.json()
        answer = result["choices"][0]["message"]["content"]
        print(f"\n[2] Inference: SUCCESS")
        print(f"    Response: {answer[:300]}")
    else:
        print(f"[2] Inference: FAILED ({resp.status_code}) — {resp.text[:200]}")
except Exception as e:
    print(f"[2] Inference FAILED: {e}")

print(f"\nNote: To test via API Gateway (local machine), set VLLM_NGROK_URL in .env first.")

Testing model server on port 8001...

INFO:     ::1:52944 - "GET /health HTTP/1.1" 200 OK
[1] Health check FAILED: Expecting value: line 1 column 1 (char 0)
INFO 05-18 10:29:53 logger.py:37] Received request chatcmpl-bc1ce3eab40b4587935d24cb23b06aee: prompt: '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nWhat is platform engineering in 2 sentences?<|im_end|>\n<|im_start|>assistant\n', params: SamplingParams(n=1, presence_penalty=0.0, frequency_penalty=0.0, repetition_penalty=1.0, temperature=0.7, top_p=1.0, top_k=-1, min_p=0.0, seed=None, stop=[], stop_token_ids=[], bad_words=[], include_stop_str_in_output=False, ignore_eos=False, max_tokens=128, min_tokens=0, logprobs=None, prompt_logprobs=None, skip_special_tokens=True, spaces_between_special_tokens=True, truncate_prompt_tokens=None, guided_decoding=None), prompt_token_ids: None, lora_request: None, prompt_adapter_request: None.
INFO 05-18 10:29:53 engine.py:273]

INFO 05-18 10:30:04 metrics.py:453] Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 5.0 tokens/s, Running: 0 reqs, Swapped: 0 reqs, Pending: 0 reqs, GPU KV cache usage: 0.0%, CPU KV cache usage: 0.0%.
INFO 05-18 10:30:14 metrics.py:453] Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Swapped: 0 reqs, Pending: 0 reqs, GPU KV cache usage: 0.0%, CPU KV cache usage: 0.0%.


## Summary

### Lab #28 Deliverables Checklist

| Step | Task | Status |
|------|------|--------|
| 1 | Docker Compose up (`docker compose up -d`) | ☐ |
| 2 | Kaggle vLLM server started | ☐ |
| 3 | ngrok tunnel created, URL copied to `.env` | ☐ |
| 4 | Prefect flows deployed | ☐ |
| 5 | Kafka data ingested (`python scripts/01_ingest_to_kafka.py`) | ☐ |
| 6 | Smoke tests passed (`pytest smoke-tests/ -v`) | ☐ |
| 7 | Production readiness >80% | ☐ |
| 8 | Screenshots captured for submission | ☐ |

### Service URLs
- API Gateway: http://localhost:8000
- Prefect UI: http://localhost:4200
- Grafana: http://localhost:3000 (admin/admin)
- Prometheus: http://localhost:9090
- Qdrant Dashboard: http://localhost:6333/dashboard